# Notebook 00 — Setup and data loading

**Purpose.** Verify your environment, load the Synapse deposit, and print sanity-check shapes. No analysis output is produced here — run this notebook first to confirm everything is wired up before working through Notebooks 01–03.

## Required Synapse files

Download from the Synapse sub-project (accession TBD). Place all files in a
single directory and set `DEPOSIT_DIR` below.

| File | Shape | Description |
|---|---|---|
| `samples_final.csv` | 546 × 276 | Component × sample loadings |
| `cell_type.csv` | 546 × 6 | Component × cell-type loadings (Ast, Oli, Inh, OPC, Exc, Mic) |
| `RNA.txt` | 25,098 × 546 | Gene × component loadings (PIP > 0.5 masked) |
| `meth.txt` | 51,798 × 546 | CpG × component loadings (PIP > 0.5 masked) |
| `chip.txt` | 26,384 × 546 | H3K9ac peak × component loadings (PIP > 0.5 masked) |
| `proteins.txt` | 5,230 × 546 | Protein × component loadings (PIP > 0.5 masked) |

## Required reference annotations

Bundled in `references/` (or downloaded yourself — see references/README.md).

| File | Description |
|---|---|
| `Homo_sapiens.GRCh38.100.txt` | Gene annotation (Ensembl 100) |
| `ROSMAP_arrayMethylation_metaData.tsv` | Illumina methylation array manifest |
| `H3K9acDomains.csv` | H3K9ac ChIP-seq peak coordinates (Klein et al.) |

Clinical metadata (`ROSMAP_clinical.csv`) is **not redistributed** — it is
controlled-access (request via the [AD Knowledge Portal DUC]
(https://adknowledgeportal.synapse.org/Data%20Access)). Notebooks that need
clinical variables document their dependency inline and degrade gracefully
when the file is absent.


## Step 1 — Imports and helper module

All notebooks share a small helper module at `g_ros_release.py` (at the repo root) that wraps deposit-file loading and the manuscript's plotting / analysis conventions. We import it here as `gr`.

In [1]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Add the repo root to sys.path so we can import the helper module.
sys.path.insert(0, "..")
import g_ros_release as gr

gr.apply_plot_style()


## Step 2 — Configure paths and load the deposit

In [2]:
# Path configuration — edit these two paths to match your local layout.
# `DEPOSIT_DIR` is where you downloaded the Synapse files (syn3219045 sub-
# project, accession TBD on publication). `REFERENCES_DIR` is the bundled
# `references/` directory in this repo.

DEPOSIT_DIR = "../synapse"   # <-- replace with your local Synapse download
REFERENCES_DIR = "../references"             # bundled with this repo

deposit = gr.load_deposit(DEPOSIT_DIR, REFERENCES_DIR, load_references=True)
print(f"Loaded {len(deposit.samples)} components × {deposit.samples.shape[1]} samples")
print(f"Cell-type matrix shape: {deposit.cell_type.shape}")
for omic, df in deposit.genes.items():
    print(f"  {omic}: {df.shape[0]} features × {df.shape[1]} components "
          f"(non-NaN cells: {int(df.notna().sum().sum()):,})")


Loaded 546 components × 276 samples
Cell-type matrix shape: (546, 6)
  RNA: 25098 features × 546 components (non-NaN cells: 278,899)
  meth: 51798 features × 546 components (non-NaN cells: 95,582)
  chip: 26384 features × 546 components (non-NaN cells: 69,918)
  proteins: 5230 features × 546 components (non-NaN cells: 6,511)


## Step 3 — Sanity checks

In [3]:
# Cell-type column order in cell_type.csv (NOT the figure plotting order)
print('cell_type.csv columns:', list(deposit.cell_type.columns))
print('Figure plotting order:', gr.CELL_TYPE_PLOT_ORDER)

# Component ID range
print(f'Components: {deposit.samples.index.min()} .. {deposit.samples.index.max()}')
print(f'Total components: {len(deposit.samples)}')


cell_type.csv columns: ['Ast', 'Oli', 'Inh', 'OPC', 'Exc', 'Mic']
Figure plotting order: ['Ast', 'Exc', 'Inh', 'Mic', 'OPC', 'Oli']
Components: 1 .. 546
Total components: 546


## Step 4 — Spot-check the headline components

These are the components featured in the published manuscript. Use `gr.HEADLINE_COMPONENTS` for the full map of component → figure label.

In [4]:
for cid, label in gr.HEADLINE_COMPONENTS.items():
    s = gr.component_summary(deposit, cid)
    print(f'Component {cid:>3} — {label}')
    print(f'    PIP-significant features: '
          f"RNA={s['n_features_RNA']}, meth={s['n_features_meth']}, "
          f"chip={s['n_features_chip']}, proteins={s['n_features_proteins']}")
    print(f'    Top cell-type drivers: +{s["top_celltype_pos"]}, -{s["top_celltype_neg"]}')


Component 351 — Figure 2 — microglial immunometabolic program
    PIP-significant features: RNA=707, meth=7, chip=584, proteins=315
    Top cell-type drivers: +Oli, -Mic
Component   2 — Figure 3 — multi-omic regulatory features
    PIP-significant features: RNA=277, meth=6, chip=7, proteins=0
    Top cell-type drivers: +Mic, -Ast
Component  74 — Figure 4 — sex-differentiated programs
    PIP-significant features: RNA=161, meth=3435, chip=676, proteins=0
    Top cell-type drivers: +Exc, -Oli
Component  87 — Figure 5 — astrocyte immune-modulation / epigenetic-reprogramming
    PIP-significant features: RNA=255, meth=2, chip=2, proteins=0
    Top cell-type drivers: +Ast, -OPC
Component 278 — Supplementary Figure S4 — female-biased OPC/precursor shift
    PIP-significant features: RNA=1114, meth=0, chip=0, proteins=0
    Top cell-type drivers: +OPC, -Oli
Component 177 — Supplementary Figure S5 — microglial bioenergetic program
    PIP-significant features: RNA=155, meth=8, chip=0, proteins

## Next steps — three analysis-type notebooks

Each downstream notebook operates on a configurable list of components (default: the six headline components above). Edit the `COMPONENTS = [...]` line near the top of each notebook to focus on a specific component or to sweep all 546.

- **Notebook 01 — Per-component panels.** Sample scatter, cell-type bar, and per-omic chromosome-scatter loadings (RNA / meth / chip / proteins). The visual building blocks of every figure in the paper.
- **Notebook 02 — Pathway enrichment.** KEGG pathway enrichment for RNA, methylation via nearest gene, and proteins. ChIP pathway enrichment is not included because peak-to-gene mapping was external to the deposit.
- **Notebook 03 — Clinical associations.** Table S1 parity tests: Mann-Whitney U for sex, Mann-Whitney U for CERAD 1/2 vs 3/4, Kruskal-Wallis for cogdx groups 1/2/4, with BH/FDR correction across all components. Requires `ROSMAP_clinical.csv` (controlled access).

**Out of scope** for these notebooks (covered in the published Methods only): GWAS / Manhattan / eQTL analyses, HOMER motif enrichment, SEA-AD cross-cohort validation. These require external tools and datasets beyond the Synapse deposit; reproducing them is described in the manuscript Methods.